# End-to-end demo — East Cameroon mining-health-risk framework

This notebook reproduces the principal results of Anaedevha et al. (2026) on the bundled
geochemical sample template.  Replace `data/geochemical/sample_template.csv` with your
harmonised compilation (same column schema) to apply the framework to a new study area.

Sections:
1. Data loading and QA/QC
2. Pollution indices (Igeo, CF, EF, PLI, RI, Nemerow PI, mCdeg)
3. Probabilistic health-risk assessment (Monte Carlo, USEPA 2011)
4. Sensitivity analysis (Spearman + Sobol')
5. Spatial autocorrelation (global Moran's I + LISA)
6. Visualisation of results

In [ ]:
import sys, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
print('Python', sys.version.split()[0], '|', platform.platform())
print('numpy', np.__version__, '| pandas', pd.__version__)

## 1. Load samples and regional background

In [ ]:
from cmhr.utils import load_geochem_table, load_config
df = load_geochem_table('../data/geochemical/sample_template.csv')
bg = load_config('reference_doses')['regional_background']
df.head()

## 2. Pollution indices

In [ ]:
from cmhr.pollution_indices import (
    geoaccumulation_index, classify_igeo,
    contamination_factor, classify_cf,
    pollution_load_index,
    ecological_risk_index, classify_ri,
    enrichment_factor, classify_ef,
    nemerow_pollution_index, classify_nemerow,
    modified_contamination_degree,
)
for el in ['As','Pb','Cd','Hg']:
    col = f'{el}_mg_kg'
    if col in df.columns:
        df[f'Igeo_{el}']  = geoaccumulation_index(df[col], bg[el])
        df[f'CF_{el}']    = contamination_factor(df[col], bg[el])
df['PLI']  = pollution_load_index(df, bg)
df['mCdeg']= modified_contamination_degree(df, bg)
df['RI']   = ecological_risk_index(df, bg)
df.filter(regex='Igeo_|CF_|PLI|mCdeg|RI').describe().T

## 3. Probabilistic health-risk assessment

In [ ]:
from cmhr.health_risk import MonteCarloRiskAssessment
mc = MonteCarloRiskAssessment(receptor='children')
results = {
    el: mc.run_ingestion(df[f'{el}_mg_l'].dropna().mean(), el, 'water')
    for el in ['As','Cd','Pb'] if df.get(f'{el}_mg_l') is not None and df[f'{el}_mg_l'].dropna().size
}
mc.batch_summary(results)

## 4. Sensitivity analysis

In [ ]:
from cmhr.health_risk import spearman_sensitivity
rng = np.random.default_rng(0)
n = 5000
inputs = pd.DataFrame({
    'C':  rng.lognormal(np.log(0.011), 0.3, n),
    'IR': rng.triangular(0.5, 0.78, 1.5, n),
    'BW': np.clip(rng.normal(15,3,n),5,30),
    'EF': rng.uniform(300,365,n),
    'ED': np.full(n,6),
})
rfd = 5e-4
hq = (inputs['C']*inputs['IR']*inputs['EF']*inputs['ED'])/(inputs['BW']*inputs['ED']*365)/rfd
spearman_sensitivity(inputs, hq.values)

## 5. Spatial autocorrelation

In [ ]:
from cmhr.geospatial import global_morans_i, local_morans_i
coords = list(zip(df['lat'], df['lon']))
moran  = global_morans_i(df['PLI'].values, coords, threshold_km=200, permutations=199)
lisa   = local_morans_i(df['PLI'].values, coords, threshold_km=200, permutations=199)
print(moran)
lisa

## 6. Visualisation

In [ ]:
from cmhr.visualization import plot_pollution_index_distribution, plot_lisa_map
fig, ax = plt.subplots(1,2, figsize=(14,5))
plot_pollution_index_distribution(df, ['PLI','RI','mCdeg'], ax=ax[0])
plot_lisa_map(lisa, df, ax=ax[1])
plt.tight_layout()